In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [5]:
spark

In [7]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [8]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [9]:
df.show(10, truncate=False)
##sql w terminalu tak wyglada

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [10]:
##nie chcemy analiuzowac tych danych w normalny ilosciowy sposbo tylko ze względu na kolumnę czasowa
##tutaj zmieniamy na typowo czasowy typ danych
##w sparku metoda withColumn (na ramce która jest modyfikuję kolumnę timestamp)
# zamiast impoty to_timestamp, col to moze byc * co oznacza ze zaladujemy wszystkie funkcje ale to nie polecane bo moze
#sie nakladac z innymi funkacjmi

from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
#agregacje - grupowanie groupby
#metoda agg
from pyspark.sql.functions import count, sum as _sum, avg, round as _round
#count - policz ile mamy transakcji i warto aliasy bo inaczej spark sam cos przydzieli
#sumy i round maj takie podkreslniki bo tak je zakodowalismy na gorze zeby odroznic ze to jest metoda ze sparka a nie z czegos innego
store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
# ta komórka uruchamia się bardzo szybko, bo ten kod tylko nam definiuje co ew spark będzie mógł zrobić jeśli wykonamy jakas akcje

In [8]:
#tutaj pokazujemu i trwa chwile dluzej
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [16]:
# Zadanie 2.2
# TWÓJ KOD
# df.groupBy("category").agg(...).orderBy("category").show()
from pyspark.sql.functions import sum as _sum, min as _min, max as _max
stats_kategorie = df.groupBy("category").agg(
    _sum("amount").alias("suma_kwot"),
    _min("amount").alias("min_kwota"),
    _max("amount").alias("max_kwota")
).orderBy("category")

stats_kategorie.show()



+-----------+------------------+---------+---------+
|   category|         suma_kwot|min_kwota|max_kwota|
+-----------+------------------+---------+---------+
|elektronika|1520770.6899999995|      9.0|   9999.0|
|    książki| 851382.0799999991|      5.0|  9107.25|
|     odzież| 849877.5500000007|      5.0|  9696.63|
|    żywność| 789514.4300000003|      5.0|  6916.92|
+-----------+------------------+---------+---------+



In [9]:
#pokazemy teraz ze bachowo mozemy przetworzyc dane w czasie
# docelowo chcemy zeby jak dane beda naplywac to my będziemy te porcje danych analizowac
#ZADANIE 3


In [10]:
#będziemy grupowac po wyniku z kolumny czyli timestampie
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne (tutaj mozemy ustalic slownie jakkolwiek chcemy)
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
#ta znowy się szybko zrealizowała 

In [11]:
hourly.show(truncate=False)
#truncate = false zeby rozszerzyc widok tego okna nizej

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [12]:
hourly.printSchema()
#w sparku w jednej kolumnie mozna zagęscic więcej elementów (nie tak jak w pythonie nie da się podzielic kolumny na dwa)
#więc ten json poczatkowy moze miec dowolna formule bo spark i tak sobie zaladuje te dane jako zagniezdzone

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [13]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)
#teraz dostajemy bardziej pandasowa ramke od do z liczba wszystkich transakcji
#mam dane z zakresu godizny, po godz nowe okno i potem znowu analizuje kolejna godzine
#za tydzien nie będzie za kazdym razem takiego samego wyniku tylko w kazdym oknie rozne wyniki bo ise bedzie na biezaco aktualizowalo


+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [13]:
##Zadanie 3.2
# TWÓJ KOD
# df.groupBy(window("timestamp", "30 minutes"), "store").agg(...).orderBy(...).show()
from pyspark.sql.functions import window, count, sum as _sum, col


okna_sklepy = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_transakcji"),
        _sum("amount").alias("suma_kwot")
    )
    .orderBy("window", "store")
)

###
(okna_sklepy.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    "store",
    "liczba_transakcji",
    "suma_kwot"
).show(truncate=False))

+-------------------+-------------------+--------+-----------------+------------------+
|od                 |do                 |store   |liczba_transakcji|suma_kwot         |
+-------------------+-------------------+--------+-----------------+------------------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252              |93391.22000000002 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289              |117786.4199999999 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275              |88441.58000000003 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296              |111540.5899999999 |
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514              |209187.85000000012|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532              |223541.41000000006|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490              |182435.05999999994|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502              |215587.1699999999 |
|2026-04-12 09:00:00|2026-04-12 

In [14]:
# Zadanie 3.3
from pyspark.sql.functions import desc

# TWÓJ KOD

wynik_krakow = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_sum("amount").alias("suma_przychodow"))
    .orderBy(desc("suma_przychodow"))
)

(wynik_krakow.select(
    col("window.start").alias("od"),
    col("window.end").alias("do"),
    "suma_przychodow"
).show(1, truncate=False))

+-------------------+-------------------+------------------+
|od                 |do                 |suma_przychodow   |
+-------------------+-------------------+------------------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.85999999975|
+-------------------+-------------------+------------------+
only showing top 1 row



In [16]:

sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min (uruchamiane co 30 min wiec nie 3 okna a 7 bd)
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)
#więcej okien, ale częśc okien zawiera te same dane
# wnioski inne niz z tamtej tabeli

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [17]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Sliding window generuje więcej wierszy, ponieważ okna nakładają się na siebie (overlap). 

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ:
#    - groupBy("store") to agregacja statyczna. Grupuje wszystkie dane 
#      z całego pliku dla każdego sklepu, ignorując czas wystąpienia transakcji.
#    - groupBy(window(...), "store") to agregacja czasowa. Dzieli dane na 
#      okresy (okna), a następnie liczy statystyki dla każdego sklepu 
#      osobno w każdym z tych przedziałów czasowych.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ:
#    Transakcja z godziny 09:30:00 wpadnie do 2 okien:
#    1.Okno [09:00, 10:00)
#    2.Okno [09:30, 10:30) - 09:30 dokładny czas rozpoczęcia tego okna.
#    Okno [08:30, 09:30) nie obejmie tej transakcji, bo prawy brzeg jest otwarty
